In [1]:
from itertools import combinations

import numpy as np
import plotly.express as px
from cogent3 import get_app, open_data_store
from cogent3.maths.measure import jsd

from clock_project.genome_analysis.correlation_analysis.triple_selection import (
    get_outgroup_species,
)

## Range of JSD

In [2]:
# from clock_project.genome_analysis.sequence_alignment_filtering import pairwise_jsd_matrix


def get_all_jsd(nuc_freqs):
    jsd_list = []
    species_keys = nuc_freqs.keys()
    for species_1, species_2 in combinations(species_keys, 2):
        jsd_value = jsd(np.array(nuc_freqs[species_1]), np.array(nuc_freqs[species_2]))
        jsd_list.append(jsd_value)
    return jsd_list

In [3]:
def quantile_calculation(values, quantile):
    values.sort()
    index = int(len(values) * quantile)
    return values[index]

In [4]:
JSD_value_range1 = []
loader = get_app("load_unaligned", format="fasta", moltype="dna")
aln_dir = "/Users/gulugulu/clock/mammal_orthologs_hsap_1/sampled_aligned"
aln_dstore = open_data_store(aln_dir, mode="w", suffix="fa")
for path in aln_dstore.completed:
    aln = loader(path)
    nuc_freqs = aln.probs_per_seq()
    pairwise_jsd = get_all_jsd(nuc_freqs)
    JSD_value_range1.extend(pairwise_jsd)

In [ ]:
# Create the histogram with density normalization

fig = px.histogram(
    [JSD_value for JSD_value in JSD_value_range1 if JSD_value != 0],
    labels={"x": "In-group JSD", "y": "Density"},
    title=None,
    # histnorm='density'  # Normalize the histogram to density
)

# Update layout for presentation
fig.update_layout(
    template="plotly_white",
    margin=dict(l=50, r=50, t=50, b=50),  # Adjust margins for a balanced look
    autosize=True,
    yaxis_title="<b>Count</b>",  # Explicit y-axis title
    xaxis_title="<b>In-group JSD</b>",  # Explicit x-axis title
    yaxis_title_font=dict(size=20),  # Adjust y-axis font size
    xaxis_title_font=dict(size=20),  # Adjust x-axis font size
    font=dict(size=16),  # General font size for labels and titles
    width=800,  # Set figure width (optional for better control)
    height=500,  # Set figure height (optional for better control)
    showlegend=False,  # Remove the legend
)

# Set transparency level and add a solid line around each bar
fig.update_traces(
    opacity=0.8,  # Set the transparency (0 = fully transparent, 1 = fully opaque)
    marker_line_color="black",  # Color of the line around each bar
    marker_line_width=1.5,  # Width of the line around each bar
)
fig.add_shape(
    type="line",
    x0=0.0206,
    y0=0,
    x1=0.0206,
    y1=18000,
    line=dict(color="red", width=3, dash="dashdot"),
)

# fig.write_image('/Users/gulugulu/Desktop/honours/figures/JSD_histogram.pdf')
fig.show()

In [6]:
JSD_value_range_dict1 = {}
loader = get_app("load_unaligned", format="fasta", moltype="dna")
aln_dir = "/Users/gulugulu/clock/mammal_orthologs_hsap_1/sampled_aligned"
aln_dstore = open_data_store(aln_dir, mode="w", suffix="fa")
for path in aln_dstore.completed:
    gene_name = path.unique_id
    aln = loader(path)
    nuc_freqs = aln.probs_per_seq()
    pairwise_jsd = get_all_jsd(nuc_freqs)
    JSD_value_range_dict1[gene_name] = pairwise_jsd

In [7]:
jsd_quantile = {
    gene: quantile_calculation(JSD_value_range_dict1[gene], 0.05)
    for gene in JSD_value_range_dict1
}


fig = px.histogram(
    jsd_quantile.values(),
    title=None,
)

# Update layout for presentation
fig.update_layout(
    template="plotly_white",
    margin=dict(l=50, r=50, t=50, b=50),  # Adjust margins for a balanced look
    autosize=True,
    yaxis_title="<b>Count</b>",  # Explicit y-axis title
    xaxis_title="<b>In-group JSD 95% quantile </b>",  # Explicit x-axis title
    yaxis_title_font=dict(size=20),  # Adjust y-axis font size
    xaxis_title_font=dict(size=20),  # Adjust x-axis font size
    font=dict(size=16),  # General font size for labels and titles
    width=800,  # Set figure width (optional for better control)
    height=500,  # Set figure height (optional for better control)
    showlegend=False,  # Remove the legend
)

# Range of ENS diff

In [8]:
load_json_app = get_app("load_json")

# # Directory and pattern to find JSON files
dir = "/Users/gulugulu/clock/mammal_orthologs_hsap_1/whole_gene_model_fitting"

# Use glob.glob to find all files matching the pattern
lf_json_files = open_data_store(dir, mode="r", suffix="json").completed

branch_length_range = []
branch_length_diff_range_dict = {}
for path in lf_json_files:
    gene_name = path.unique_id
    branch_length_diff_range_dict[gene_name] = []
    lf = load_json_app(path)
    ens_tree = lf.lf.get_ens_tree()
    edge_names = lf.tree.get_tip_names()
    for comb in combinations(edge_names, 2):
        if comb[0] != comb[1]:
            ingroup_species_pair = [comb[0], comb[1]]
            outgroup_species = get_outgroup_species(ingroup_species_pair, ens_tree)
            if outgroup_species:
                triples_species_names = {
                    "ingroup1": ingroup_species_pair[0],
                    "ingroup2": ingroup_species_pair[1],
                    "outgroup": outgroup_species[0],
                }
                try:
                    sub_tree = ens_tree.get_sub_tree(triples_species_names.values())
                    ens_dict = {
                        n: sub_tree.to_rich_dict()["edge_attributes"][n]["length"]
                        for n in triples_species_names.values()
                    }
                    ens_difference = np.sqrt(
                        (
                            ens_dict[ingroup_species_pair[0]]
                            - ens_dict[ingroup_species_pair[1]]
                        )
                        ** 2
                    )
                    branch_length_range.append(ens_difference)
                    branch_length_diff_range_dict[gene_name].append(ens_difference)
                except Exception as e:
                    print(e)

/Users/gulugulu/miniconda3/envs/c312/lib/python3.12/site-packages/cogent3/recalculation/definition.py:690: UserWarning:

using slow exponentiator because 'eigen failed precision test'



In [12]:
# Create the histogram with density normalization
fig = px.histogram(
    branch_length_range,
    labels={"x": "Branch length difference", "y": "Density"},
    title=None,
    # histnorm='density'  # Normalize the histogram to density
)

# Update layout for presentation
fig.update_layout(
    template="plotly_white",
    margin=dict(l=50, r=50, t=50, b=50),  # Adjust margins for a balanced look
    autosize=True,
    yaxis_title="<b>Count</b>",  # Explicit y-axis title
    xaxis_title="<b>Branch length difference</b>",  # Explicit x-axis title
    yaxis_title_font=dict(size=20),  # Adjust y-axis font size
    xaxis_title_font=dict(size=20),  # Adjust x-axis font size
    font=dict(size=16),  # General font size for labels and titles
    width=800,  # Set figure width (optional for better control)
    height=500,  # Set figure height (optional for better control)
    showlegend=False,  # Remove the legend
)

# Set transparency level and add a solid line around each bar
fig.update_traces(
    opacity=0.8,  # Set the transparency (0 = fully transparent, 1 = fully opaque)
    marker_line_color="black",  # Color of the line around each bar
    marker_line_width=1.5,  # Width of the line around each bar
)
# fig.add_shape(
#     type="line",
#     x0=0.23, y0=0, x1=0.23, y1=7700,
#     line=dict(color="red", width=3, dash="dashdot"),
# )

fig.show()

In [10]:
quantile_calculation(branch_length_range, 0.95)

0.29467314679456674

In [11]:
branch_length_quantile = {
    gene: quantile_calculation(branch_length_diff_range_dict[gene], 0.95)
    for gene in branch_length_diff_range_dict
}


fig1 = px.histogram(branch_length_quantile.values())
# Update layout for presentation
fig1.update_layout(
    template="plotly_white",
    margin=dict(l=50, r=50, t=50, b=50),  # Adjust margins for a balanced look
    autosize=True,
    yaxis_title="<b>Count</b>",  # Explicit y-axis title
    xaxis_title="<b>Banch length differnece 95% quantile </b>",  # Explicit x-axis title
    yaxis_title_font=dict(size=20),  # Adjust y-axis font size
    xaxis_title_font=dict(size=20),  # Adjust x-axis font size
    font=dict(size=16),  # General font size for labels and titles
    width=800,  # Set figure width (optional for better control)
    height=500,  # Set figure height (optional for better control)
    showlegend=False,  # Remove the legend
)